In [ ]:
"""Setup: load YAML data + flat alt_df, derive helper bindings used by every chart cell.

The shared module `prompt_analysis.py` lives next to this notebook in the project root.
"""
import importlib
import altair as alt
import pandas as pd

import prompt_analysis
importlib.reload(prompt_analysis)   # pick up edits without restarting the kernel
from prompt_analysis import (
    load_yaml, build_alt_df, version_order, category_colors,
    directiveness,
    SR_CLASS_COLORS, SENT_REGISTER_CLASSES, TABLEAU10,
)

alt.data_transformers.disable_max_rows()

data              = load_yaml()                  # default: prompt_linguistic_analysis.yaml
alt_df            = build_alt_df(data)
by_category       = data["by_category"]
corpus_block      = data["corpus"]
per_file_records  = data["files"]
cats              = list(by_category.keys())
VOCAB_KEYS        = list(data["lexicons"]["VOCAB"].keys())

# Composite directiveness column — used by 05 and 06.
alt_df["directiveness"] = directiveness(alt_df)

# Per-category palette + Altair encodings used across charts.
CATEGORY_COLORS = category_colors(cats)
_cat_domain     = cats
_cat_range      = [CATEGORY_COLORS[c] for c in cats]

print(f"loaded {len(per_file_records)} files | {alt_df.shape[1]} columns | {len(cats)} categories | {len(VOCAB_KEYS)} VOCAB keys")


# Emphasis, CAPS imperative, and full vocabulary profile

Three views of how authoritative the prompts are at the surface level:

1. **Emphasis density per category** — ALL CAPS density, CAPS-imperative density, and justification ratio side-by-side (independent x-scales since they live on different magnitudes).
2. **Per-file outliers (text)** — top files by CAPS-imperative density, hard-prohibitions density, and justification ratio. Bash-sandbox tool descriptions tend to dominate the top of these lists.
3. **Top tokens + full VOCAB heatmap** — the top-25 ALL CAPS tokens, all CAPS-imperative tokens, and an 11-class VOCAB profile (% of file tokens) per category.


In [ ]:
"""Emphasis: ALL CAPS, CAPS imperative, justification ratio per category — Altair."""

emphasis_long = pd.DataFrame([
    {"category": cat, "metric": metric, "value": value}
    for cat in cats
    for metric, value in [
        ("ALL CAPS (% tokens)",
            by_category[cat]["metrics"]["all_caps"]["pct"]),
        ("CAPS imperative (% tokens)",
            by_category[cat]["metrics"]["caps_imperative"]["pct"]),
        ("Justification ratio",
            by_category[cat]["metrics"]["justification"]["ratio"]),
    ]
])

emphasis_chart = (
    alt.Chart(emphasis_long)
    .mark_bar()
    .encode(
        x=alt.X("value:Q", title=None),
        y=alt.Y("category:N", sort=cats, title=None),
        color=alt.Color("category:N",
                         scale=alt.Scale(domain=cats,
                                         range=[CATEGORY_COLORS[c] for c in cats]),
                         legend=None),
        column=alt.Column("metric:N", title=None,
                            sort=["ALL CAPS (% tokens)",
                                  "CAPS imperative (% tokens)",
                                  "Justification ratio"]),
        tooltip=[alt.Tooltip("category:N"),
                 alt.Tooltip("metric:N"),
                 alt.Tooltip("value:Q", format=".3f")],
    )
    .resolve_scale(x="independent")
    .properties(width=240, height=240,
                title="Emphasis density per category (independent x-scales)")
)
emphasis_chart


In [ ]:
"""Per-file outliers: highest CAPS-imperative density and lowest justification ratio."""

per_file_df = pd.DataFrame([
    {
        "path": r["path"],
        "category": r["category"],
        "n_tokens": r["n_tokens"],
        "imp_sent_pct":     r["metrics"]["sentence_register"]["imperative_sent_pct"],
        "caps_imp_pct":     r["metrics"]["caps_imperative"]["pct"],
        "all_caps_pct":     r["metrics"]["all_caps"]["pct"],
        "just_ratio":       r["metrics"]["justification"]["ratio"],
        "deontic_pct":      r["metrics"]["modality"]["deontic_pct"],
        "hard_proh_pct":    r["metrics"]["vocab"]["hard_prohibitions"]["pct"],
    }
    for r in per_file_records
])

print("--- 10 files with highest CAPS-imperative density (% of file tokens) ---")
print(per_file_df.nlargest(10, "caps_imp_pct")[["path", "category", "n_tokens", "caps_imp_pct"]].to_string(index=False))
print("\n--- 10 files with highest hard_prohibitions density (% of file tokens) ---")
print(per_file_df.nlargest(10, "hard_proh_pct")[["path", "category", "n_tokens", "hard_proh_pct"]].to_string(index=False))
print("\n--- 10 files with most explanatory tone (highest justification ratio, ≥150 tokens) ---")
big = per_file_df[per_file_df["n_tokens"] >= 150]
print(big.nlargest(10, "just_ratio")[["path", "category", "n_tokens", "just_ratio"]].to_string(index=False))


### Emphasis vocabulary: top ALL CAPS tokens, CAPS imperative tokens, and full VOCAB profile

In [ ]:
"""Top-N tokens for ALL CAPS and CAPS imperative + full VOCAB heatmap."""

top_caps = pd.DataFrame(corpus_block["metrics"]["all_caps"]["top"][:25])
top_caps_chart = (
    alt.Chart(top_caps)
    .mark_bar(color="#af7aa1")
    .encode(
        x=alt.X("count:Q", title="corpus-wide count"),
        y=alt.Y("token:N", sort="-x", title=None),
        tooltip=[alt.Tooltip("token:N"), alt.Tooltip("count:Q")],
    )
    .properties(width=320, height=380,
                title="Top 25 ALL CAPS tokens (TECH_ACRONYMS excluded)")
)

caps_imp_data = pd.DataFrame(
    [{"token": t, "count": c} for t, c in
     corpus_block["metrics"]["caps_imperative"]["hits"].items()]
)
caps_imp_chart = (
    alt.Chart(caps_imp_data)
    .mark_bar(color="#e15759")
    .encode(
        x=alt.X("count:Q", title="corpus-wide count"),
        y=alt.Y("token:N", sort="-x", title=None),
        tooltip=[alt.Tooltip("token:N"), alt.Tooltip("count:Q")],
    )
    .properties(width=320, height=380,
                title="CAPS imperative tokens (corpus-wide)")
)

vocab_long = []
for cat, b in by_category.items():
    for key, v in b["metrics"]["vocab"].items():
        vocab_long.append({"category": cat, "vocab_key": key, "pct": v["pct"]})
vocab_df_long = pd.DataFrame(vocab_long)

vocab_chart = (
    alt.Chart(vocab_df_long)
    .mark_rect()
    .encode(
        x=alt.X("vocab_key:N", title=None, sort=list(VOCAB_KEYS)),
        y=alt.Y("category:N", title=None),
        color=alt.Color("pct:Q", scale=alt.Scale(scheme="magma", reverse=True),
                         title="% of file tokens"),
        tooltip=[alt.Tooltip("category:N"),
                 alt.Tooltip("vocab_key:N"),
                 alt.Tooltip("pct:Q", format=".3f")],
    )
    .properties(width=720, height=260,
                title="Full VOCAB profile per category (% of file tokens)")
)

(top_caps_chart | caps_imp_chart) & vocab_chart
